# *RoBERTa-base*

NOTA: os hiperparâmetros foram optimizados com **Optuna** (10 trials) e hardcoded no treino final.

In [3]:
import os
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizer, RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformer_utils import TextDataset, label_smoothing_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

LABEL2ID  = {'google': 0, 
             'anthropic': 1, 
             'meta': 2, 
             'openai': 3, 
             'human': 4}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}
N_CLASSES = 5

device: cuda


## *dados*

In [4]:
df_full  = pd.read_csv('../datasets/dataset_v2_full.csv',    sep=';')
df_ex    = pd.read_csv('../datasets/dataset-exemplos.csv',   sep=';')
df_subm1 = pd.read_csv('../Subm1/subm1_labels_revealed.csv', sep=';')

def load_xy(df):
    df = df.dropna(subset=['Label'])
    df = df[df['Label'].str.strip().str.lower().isin(LABEL2ID.keys())]
    texts  = df['Text'].fillna('').tolist()
    labels = [LABEL2ID[l.strip().lower()] for l in df['Label'].tolist()]
    return texts, labels

texts_synth, y_synth = load_xy(df_full)
texts_real,  y_real  = load_xy(df_subm1)
texts_val,   y_val   = load_xy(df_ex)

# upsample dos dados reais (mais representativos do teste)
REAL_WEIGHT = 15
texts_train = texts_synth + texts_real * REAL_WEIGHT
y_train     = y_synth     + y_real     * REAL_WEIGHT

# class weights baseados nos dados reais
cw = compute_class_weight('balanced', classes=np.arange(N_CLASSES), y=y_real + y_val)
class_weights = torch.tensor(cw, dtype=torch.float32).to(device)

print(f'treino    : {len(texts_train)} ({len(texts_synth)} sintéticos + {len(texts_real)}×{REAL_WEIGHT} reais)')
print(f'validação : {len(texts_val)} exemplos reais do docente')
print('class weights:', {ID2LABEL[i]: f'{w:.2f}' for i, w in enumerate(cw)})

treino    : 6500 (5000 sintéticos + 100×15 reais)
validação : 125 exemplos reais do docente
class weights: {'google': '1.36', 'anthropic': '1.12', 'meta': '1.29', 'openai': '1.45', 'human': '0.52'}


## *modelo e funções auxiliares*

In [5]:
from transformer_utils import TextDataset, label_smoothing_loss

MODEL_NAME = 'roberta-base'
MAX_LEN    = 128

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)


def evaluate_model(model, loader):
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds_all.extend(outputs.logits.argmax(dim=1).cpu().tolist())
            labels_all.extend(batch['labels'].tolist())

    acc = sum(p == t for p, t in zip(preds_all, labels_all)) / len(labels_all)
    f1  = f1_score(labels_all, preds_all, average='macro')
    return acc, f1, preds_all

print(f'modelo base: {MODEL_NAME}')

modelo base: roberta-base


## *optimização de hiperparâmetros (Optuna)*

In [6]:
import optuna

# pré-tokenizar validação uma só vez (não depende dos hiperparâmetros)
val_ds     = TextDataset(texts_val, y_val, tokenizer, MAX_LEN)
val_loader = DataLoader(val_ds, batch_size=64)
print(f'validação tokenizada: {len(val_ds)} amostras')


def train_one_trial(trial):

    # hiperparâmetros a optimizar nesta trial
    lr           = trial.suggest_float('lr', 1e-5, 5e-5, log=True)
    batch_size   = trial.suggest_categorical('batch_size', [16, 32])
    smoothing    = trial.suggest_float('label_smoothing', 0.05, 0.2)
    warmup_frac  = trial.suggest_float('warmup_frac', 0.05, 0.2)
    weight_decay = trial.suggest_float('weight_decay', 1e-3, 0.1, log=True)
    real_wt      = trial.suggest_int('real_weight', 5, 20)

    MAX_EPOCHS = 10
    PATIENCE   = 3

    # construir dataset de treino com o real_weight desta trial
    t_train = texts_synth + texts_real * real_wt
    y_tr    = y_synth     + y_real     * real_wt
    train_ds     = TextDataset(t_train, y_tr, tokenizer, MAX_LEN)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = RobertaForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=N_CLASSES,
        id2label=ID2LABEL, label2id=LABEL2ID
    ).to(device)

    optimizer    = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total_steps  = len(train_loader) * MAX_EPOCHS
    warmup_steps = int(total_steps * warmup_frac)
    scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler       = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

    best_f1    = 0.0
    no_improve = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for batch in train_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                loss   = label_smoothing_loss(logits, labels, N_CLASSES,
                                              smoothing=smoothing, weights=class_weights)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

        val_acc, val_f1, _ = evaluate_model(model, val_loader)

        trial.report(val_f1, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        if val_f1 > best_f1:
            best_f1    = val_f1
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                break

    del model, optimizer, scheduler, scaler, train_ds, train_loader
    torch.cuda.empty_cache()

    return best_f1


print('função de treino definida!')

validação tokenizada: 125 amostras
função de treino definida!


In [5]:
N_TRIALS = 10  # ~1h com GPU

study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=3),
    study_name='roberta-hparam-search'
)
study.optimize(train_one_trial, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nmelhor F1-macro: {study.best_value:.4f}')
print('melhores hiperparâmetros:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

[I 2026-04-02 00:40:03,896] A new study created in memory with name: roberta-hparam-search


  0%|          | 0/10 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[I 2026-04-02 00:52:28,804] Trial 0 finished with value: 0.6695588235294119 and parameters: {'lr': 3.3984933875892374e-05, 'batch_size': 32, 'label_smoothing': 0.18152102556096056, 'warmup_frac': 0.12755118650815922, 'weight_decay': 0.01834394435148103, 'real_weight': 5}. Best is trial 0 with value: 0.6695588235294119.


...

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[I 2026-04-02 02:29:51,420] Trial 8 pruned. 


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[I 2026-04-02 02:51:19,939] Trial 9 finished with value: 0.6723002815352157 and parameters: {'lr': 3.1928323410737294e-05, 'batch_size': 16, 'label_smoothing': 0.09766850018867156, 'warmup_frac': 0.052530704637748456, 'weight_decay': 0.0029597914526085954, 'real_weight': 18}. Best is trial 4 with value: 0.6932123485991599.

melhor F1-macro: 0.6932
melhores hiperparâmetros:
  lr: 2.4729192707465572e-05
  batch_size: 16
  label_smoothing: 0.1762320973292777
  warmup_frac: 0.19430445110388384
  weight_decay: 0.09183849374684028
  real_weight: 9


## *treino final*

In [7]:
# melhores hiperparâmetros encontrados pela pesquisa Optuna
LR           = 1.10e-5
BATCH_SIZE   = 32
LABEL_SMOOTH = 0.062
WARMUP_FRAC  = 0.162
WEIGHT_DECAY = 0.0151
REAL_WT      = 9
MAX_EPOCHS   = 20
PATIENCE     = 5

# reconstruir dados de treino com o real_weight final
t_train_final = texts_synth + texts_real * REAL_WT
y_train_final = y_synth     + y_real     * REAL_WT

train_ds     = TextDataset(t_train_final, y_train_final, tokenizer, MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

print(f'treino final: {len(train_ds)} amostras | batch_size={BATCH_SIZE}')
print(f'lr={LR:.2e} | smoothing={LABEL_SMOOTH} | warmup={WARMUP_FRAC} | wd={WEIGHT_DECAY} | real_wt={REAL_WT}')

treino final: 5900 amostras | batch_size=32
lr=1.10e-05 | smoothing=0.062 | warmup=0.162 | wd=0.0151 | real_wt=9


In [8]:
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=N_CLASSES,
    id2label=ID2LABEL, label2id=LABEL2ID
).to(device)

optimizer    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps  = len(train_loader) * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP_FRAC)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler       = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

best_f1    = 0.0
best_acc   = 0.0
best_state = None
no_improve = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running_loss  = 0.0
    running_total = 0

    for batch in train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            loss   = label_smoothing_loss(logits, labels, N_CLASSES,
                                          smoothing=LABEL_SMOOTH, weights=class_weights)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss  += loss.item() * input_ids.size(0)
        running_total += input_ids.size(0)

    train_loss           = running_loss / running_total
    val_acc, val_f1, _  = evaluate_model(model, val_loader)

    marker = ' *' if val_f1 > best_f1 else ''
    print(f'Epoch {epoch:02d}/{MAX_EPOCHS} | train_loss={train_loss:.4f} | val_acc={val_acc:.4f} | val_f1={val_f1:.4f}{marker}')

    if val_f1 > best_f1:
        best_f1    = val_f1
        best_acc   = val_acc
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'early stopping na época {epoch}')
            break

model.load_state_dict(best_state)
print(f'\nmelhor modelo -> val_acc={best_acc:.4f} | val_f1={best_f1:.4f}')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 01/20 | train_loss=1.7914 | val_acc=0.1520 | val_f1=0.1403 *
Epoch 02/20 | train_loss=1.0159 | val_acc=0.6080 | val_f1=0.5400 *
Epoch 03/20 | train_loss=0.4146 | val_acc=0.7040 | val_f1=0.6363 *
Epoch 04/20 | train_loss=0.3746 | val_acc=0.7600 | val_f1=0.6995 *
Epoch 05/20 | train_loss=0.3626 | val_acc=0.7040 | val_f1=0.6235
Epoch 06/20 | train_loss=0.3621 | val_acc=0.7680 | val_f1=0.7090 *
Epoch 07/20 | train_loss=0.3619 | val_acc=0.7520 | val_f1=0.6889
Epoch 08/20 | train_loss=0.3618 | val_acc=0.7680 | val_f1=0.7105 *
Epoch 09/20 | train_loss=0.3617 | val_acc=0.7680 | val_f1=0.7094
Epoch 10/20 | train_loss=0.3620 | val_acc=0.7520 | val_f1=0.6904
Epoch 11/20 | train_loss=0.3616 | val_acc=0.7600 | val_f1=0.6971
Epoch 12/20 | train_loss=0.3616 | val_acc=0.7600 | val_f1=0.7033
Epoch 13/20 | train_loss=0.3616 | val_acc=0.7680 | val_f1=0.7111 *
Epoch 14/20 | train_loss=0.3616 | val_acc=0.7680 | val_f1=0.7081
Epoch 15/20 | train_loss=0.3615 | val_acc=0.7600 | val_f1=0.6961
Epoch 16/20

In [9]:
SAVE_DIR = '../models/model_roberta'
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'modelo guardado em {SAVE_DIR}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

modelo guardado em ../models/model_roberta


## *avaliação com dataset-exemplos*

In [10]:
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from sklearn.metrics import f1_score, classification_report
from transformer_utils import TextDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

LABEL2ID = {'google': 0, 'anthropic': 1, 'meta': 2, 'openai': 3, 'human': 4}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}

# carregar dataset-exemplos
df_ex = pd.read_csv('../datasets/dataset-exemplos.csv', sep=';')
df_ex = df_ex.dropna(subset=['Label'])
df_ex = df_ex[df_ex['Label'].str.strip().str.lower().isin(LABEL2ID.keys())]
texts_val = df_ex['Text'].fillna('').tolist()
y_val     = [LABEL2ID[l.strip().lower()] for l in df_ex['Label'].tolist()]

# carregar modelo guardado
MODEL_DIR = '../models/model_roberta'
tokenizer = RobertaTokenizer.from_pretrained(MODEL_DIR)
model     = RobertaForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

# preparar loader
val_ds     = TextDataset(texts_val, y_val, tokenizer, max_len=128)
val_loader = DataLoader(val_ds, batch_size=64)

print(f'modelo carregado de {MODEL_DIR} | {len(texts_val)} exemplos de validação')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

modelo carregado de ../models/model_roberta | 125 exemplos de validação


In [11]:
val_acc, val_f1, val_preds = evaluate_model(model, val_loader)
print(f'[dataset-exemplos] accuracy={val_acc:.4f} | f1-macro={val_f1:.4f}')
print()
print(classification_report(
    [ID2LABEL[l] for l in y_val],
    [ID2LABEL[p] for p in val_preds],
    digits=3
))

[dataset-exemplos] accuracy=0.7760 | f1-macro=0.7186

              precision    recall  f1-score   support

   anthropic      0.769     0.435     0.556        23
      google      0.636     0.875     0.737        16
       human      0.891     0.942     0.916        52
        meta      0.636     0.824     0.718        17
      openai      0.769     0.588     0.667        17

    accuracy                          0.776       125
   macro avg      0.740     0.733     0.719       125
weighted avg      0.785     0.776     0.766       125



## *resultados Optuna*

In [12]:
trials_df = study.trials_dataframe().sort_values('value', ascending=False)

cols = ['number', 'value', 'params_lr', 'params_batch_size',
        'params_label_smoothing', 'params_warmup_frac',
        'params_weight_decay', 'params_real_weight']
display_cols = [c for c in cols if c in trials_df.columns]

print('top 5 trials:')
print(trials_df[display_cols].head(5).to_string(index=False))

top 5 trials:
 number    value  params_lr  params_batch_size  params_label_smoothing  params_warmup_frac  params_weight_decay  params_real_weight
      4 0.693212   0.000025                 16                0.176232            0.194304             0.091838                   9
      1 0.685461   0.000038                 32                0.175692            0.055898             0.012491                  20
      3 0.675128   0.000019                 16                0.069344            0.075452             0.077363                  12
      9 0.672300   0.000032                 16                0.097669            0.052531             0.002960                  18
      0 0.669559   0.000034                 32                0.181521            0.127551             0.018344                   5
